# Langchain Chatbot

### Install requirements

In [1]:
%pip install -U langchain langchain-community pypdf pytube langchain-openai openai python-dotenv youtube-transcript-api yt_dlp pydub chromadb

Note: you may need to restart the kernel to use updated packages.


### Imports and Envrionment

In [2]:
import os
import openai
import sys

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

api_key = os.environ.get("OPENAI_API_KEY")

## Document Loading

Document loaders access data from a variety of data sources and load them into a standard object. There are a lot of data loaders in Langchain (e.g., YouTube, PowerPoint, Figma, Notion, Excel, Pandas, Amazon AWS).

### Load PDF Documents

First let's load some scientific pdfs we downloaded from the web about marathon running. 

In [3]:
from langchain.document_loaders import PyPDFLoader

PDF_DIR = 'pdfs'

pdf_pages = []

print(os.getcwd())

for filename in os.listdir(PDF_DIR):
    filepath = os.path.join(PDF_DIR, filename)
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    pdf_pages.extend(pages)

print(f"Loaded {len(pdf_pages)} pages.")

# Inspect first page
page = pdf_pages[0]
print(page.page_content[:500])
print(page.metadata)

/Users/jonas/Repositories/jonas-ml-lab/notebooks/projects/marathon coach
Loaded 250 pages.
Vol.:(0123456789)
Sports Medicine (2024) 54:1801–1833 
https://doi.org/10.1007/s40279-024-02018-z
SYSTEMATIC REVIEW
The Effect of Strength Training Methods on Middle‑Distance 
and Long‑Distance Runners’ Athletic Performance: A Systematic 
Review with Meta‑analysis
Cristian Llanos‑Lagos1  · Rodrigo Ramirez‑Campillo2 · Jason Moran3 · Eduardo Sáez de Villarreal1
Accepted: 10 March 2024 / Published online: 17 April 2024 
© The Author(s) 2024
Abstract
Background The running performance of middle-dist
{'producer': 'PyPDF', 'creator': 'Springer', 'creationdate': '2024-04-16T17:19:55+05:30', 'author': 'Cristian Llanos-Lagos', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2024-07-10T21:13:03+05:30', 'subject': 'Sports Medicine, https://doi.org/10.1007/s40279-024-02018-z', 'title':

In [4]:
from langchain_community.document_loaders import YoutubeLoader, YoutubeAudioLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

AUDIO_DIR = "audio"

URLS = [
    "https://www.youtube.com/watch?v=QEtIxBL497U",
]

# Pro: Fast and no API costs; Con: Only works if the video has transcripts
# def load_via_youtube_api(url: str):
#     loader = YoutubeLoader.from_youtube_url(
#         url, add_video_info=False, language=["en", "en-US"]
#     )
#     return loader.load()

# Uses OpenAI Whisper API (paid). Requires OPENAI_API_KEY and ffmpeg installed.
def load_via_OpenAI(url: str):
    audio_loader = YoutubeAudioLoader([url], AUDIO_DIR)
    parser = OpenAIWhisperParser()  # uses OPENAI_API_KEY from env
    loader = GenericLoader(audio_loader, parser)
    return loader.load()

yt_pages = []
for url in URLS:
    docs = load_via_OpenAI(url)
    yt_pages.extend(docs)

print(f"Loaded {len(yt_pages)} pages from YouTube.")

# Inspect first page (if any)
if yt_pages:
    page = yt_pages[0]
    print(page.page_content[:500])
    print(page.metadata)
else:
    print("No content loaded. Check URLs, transcripts availability, or Whisper setup.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=QEtIxBL497U
[youtube] QEtIxBL497U: Downloading webpage
[youtube] QEtIxBL497U: Downloading tv simply player API JSON
[youtube] QEtIxBL497U: Downloading tv client config
[youtube] QEtIxBL497U: Downloading player 9bae1a63-main
[youtube] QEtIxBL497U: Downloading tv player API JSON
[info] QEtIxBL497U: Downloading 1 format(s): 140
[download] audio/The Smartest Way to Run a Faster Marathon (Science Explained).m4a has already been downloaded
[download] 100% of   17.53MiB
[ExtractAudio] Not converting audio audio/The Smartest Way to Run a Faster Marathon (Science Explained).m4a; file is already in target format m4a
Transcribing part 1!
Loaded 1 pages from YouTube.
Most marathon advice online is outdated or flat-out wrong. I'm a sports scientist, physiotherapist, and former professional triathlete. And in this video, I'll show you exactly how to train, fuel, pace, and plan your week to get the fastest time possible. Based on real science,

In [5]:
from langchain.document_loaders import WebBaseLoader

urls = [
    "https://run.outsideonline.com/training/training-plans/marathon/a-16-week-marathon-training-plan-to-go-the-distance?scope=anon"
]

loader = WebBaseLoader(urls)
web_pages = loader.load()

print(f"Loaded {len(web_pages)} webpages.")
print(web_pages[0].page_content[:5])
print(web_pages[0].metadata)  # has source URL, title (if found), etc.

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 webpages.






{'source': 'https://run.outsideonline.com/training/training-plans/marathon/a-16-week-marathon-training-plan-to-go-the-distance?scope=anon', 'title': '16 Week Marathon Training Plan to Go the Distance', 'description': "If you're going to run a 26.2-mile race, you'll need a marathon training plan that incorporates proper strength training and recovery.", 'language': 'en-US'}


## Document splitting (Chunking)

It is important to keep meaningful relationsships! 

**Example**

- Chunk 1: The Iphone 15 has a 
- Chunk 2: 48 MP camera
- Chunk 3: The Iphone 13 has a
- Chunk 4: 12 MP camera

How many MP has the Iphone 15? 

### Langchain Text Splitters

**CharacterTextSplitter**: Split documents by a fix `chunk_size` and `chunk_overlap`.

**MarkdownHeaderTextSplitter**: Split markdown files based on specified headers.

**TokenTextSplitter**: Split text by tokens.

**SentenceTransformersTokenTextSplitter**: Split text by tokens using SentenceTransformers.

**RecursiveCharacterTextSplitter**: Split text by characters, recursively trying different split points until a good one is found.

**Language**: Split code files (e.g., C++, Python, Ruby, Markdown).

**NLTKTextSplitter**: Split text into sentences using NLTK (Natural Language Toolkit).

**SpacyTextSplitter**: Split text into sentences using SpaCy.

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

chunk_size = 26 
chunk_overlap = 4

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)


text1 = 'abcdefghijklmnopqrstuvwxyz'
text2 = 'a b c d e f g h i j k l m o p q r s t u v w x y z'

print('Example: Recursive Character Text Splitter')
print(r_splitter.split_text(text1))
print(r_splitter.split_text(text1+text1))

c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator = ' ', # Default new line.
)

print('Example: Recursive Character Text Splitter')
# It will not split because it splits on a specific character.
print(c_splitter.split_text(text1+text1)) 
# It splits after position 26 at the new space character
print(c_splitter.split_text(text2))

Example: Recursive Character Text Splitter
['abcdefghijklmnopqrstuvwxyz']
['abcdefghijklmnopqrstuvwxyz', 'wxyzabcdefghijklmnopqrstuv', 'stuvwxyz']
Example: Recursive Character Text Splitter
['abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz']
['a b c d e f g h i j k l m', 'l m o p q r s t u v w x y', 'x y z']


**Seperators**

In more real world scenarios it is important to add some good seperators to the splitters otherwise we won't receive good chunks. 

In [7]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=0,
    separators=[
        "\n\n",     # split at double newlines (paragraphs)
        "\n",       # split at single newline (lines)
        r"(?<=\.)", # split right after a period (sentence boundaries)
        " ",        # split at spaces (words)
        ""          # fallback: character-level splitting
    ]
)

splitted_web_pages = r_splitter.split_documents(web_pages)
print(f"Splitted the web page into {len(splitted_web_pages)} chunks.")
print(f"Content of the first chunk:\n{splitted_web_pages[0]}")

Splitted the web page into 213 chunks.
Content of the first chunk:
page_content='16 Week Marathon Training Plan to Go the Distance' metadata={'source': 'https://run.outsideonline.com/training/training-plans/marathon/a-16-week-marathon-training-plan-to-go-the-distance?scope=anon', 'title': '16 Week Marathon Training Plan to Go the Distance', 'description': "If you're going to run a 26.2-mile race, you'll need a marathon training plan that incorporates proper strength training and recovery.", 'language': 'en-US'}


Next let us experiment with a **TokenTextSplitter**.

In [8]:
from langchain.text_splitter import TokenTextSplitter

t_splitter = TokenTextSplitter(
    chunk_size = 1,
    chunk_overlap = 0,
)

text3 = "foo bar foobarzy"

t_splitter.split_text(text3)

['foo', ' bar', ' fo', 'obar', 'zy']

Finally let's split our pdf data for the next chapter.

In [9]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

pdf_splits = r_splitter.split_documents(pdf_pages)
len(pdf_splits)

792

# Vector Stores and Embeddings

We now use OpenAI to create Embeddings

Embedding vectors capture content and meaning of text. 

We store this vectors in a vector database. 

Later we pick the n-most similar vectors to a user prompt and augment the original prompt by our findings before we pass it back to the LLM.

In [11]:
from langchain.embeddings.openai import OpenAIEmbeddings

# Choose an embeddings model:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

sentence1 = 'I like cheesecake.'
sentence2 = 'I like ice creame.'
sentence3 = 'The weather is cloudy.'

# 💸 billable API callsn using OPENAI_API_KEY
emb1 = embeddings.embed_query(sentence1)
emb2 = embeddings.embed_query(sentence2)
emb3 = embeddings.embed_query(sentence3)

In [12]:
from sklearn.metrics.pairwise import euclidean_distances as dist

print(dist([emb1], [emb2]))  # distance between sentence1 and sentence2
print(dist([emb1], [emb3]))  # distance between sentence1 and sentence3

[[0.94127296]]
[[1.32405499]]


We now include these vectors into the **Chroma** in memory vector database.

In [13]:
from langchain.vectorstores import Chroma

CHROMA_DIR = 'chroma'

In [14]:
%rm -rf ./chroma

In [16]:
vectordb = Chroma.from_documents(
    documents=pdf_splits[:50],
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)

In [17]:
print(vectordb._collection.count()) 

50


Now let's do a similarity search on the database.

In [18]:
question = "How can i improve my V02max?"

docs = vectordb.similarity_search(question, k=3)


print(f"Document1: {docs[0].page_content[:150]}")
print(f"Document2: {docs[1].page_content[:150]}")

Document1: 1824 C. Llanos-Lagos et al.
4.1  VO2max and vVO2max
Maximal oxygen uptake is defined as the highest rate at 
which oxygen can be taken up and utilized
Document2: 1825
Strength Training in Endurance Runners
which include all physiological pathways that transfer oxy-
gen from the environment to the blood, pumping


### Maximum marginal relevance (MMR) 

Problem: When you always choose the documents that are most similar to the response. You might miss out on important context information.

1. We overfetch similar documents.
2. We rerank them based on relevance and **diversity**.

In [19]:
docs = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=9)

print(f"Document1: {docs[0].page_content[:150]}")
print(f"Document2: {docs[1].page_content[:150]}")

Document1: 1824 C. Llanos-Lagos et al.
4.1  VO2max and vVO2max
Maximal oxygen uptake is defined as the highest rate at 
which oxygen can be taken up and utilized
Document2: MMSS (e.g., second lactate threshold) establishes the limit 
of steady-state muscle metabolism [20]. Running economy, 
defined as the amount of energy


As you can see we retrieved different documents as before. However, as the docs are not well formatted and information is maybe still redundant or doubled, we can add LLM based document compression:

In [20]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI

# carefull 💸 billable API calls using OPENAI_API_KEY
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# build retriever with MMR as  before
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 15}
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(question)

print(f"Document1: {compressed_docs[0].page_content}")
print(f"Document2: {compressed_docs[1].page_content}")

Document1: Maximal oxygen consumption, velocity at maximal oxygen consumption, maximum metabolic steady state, and sprint capacity exhibited trivial changes after strength training. Indeed, VO2max has long been used as a primary measure of an individual’s cardiorespiratory fitness, and as a marker of training effect. The interplay between VO2max and running economy determines vVO2max.
Document2: The implementation of strength training (ST) can improve the performance in middle-distance and long-distance runners [10–14]. However, previous meta-analyses have focused mainly on running economy [11–13] and time trial running performance [13], without exploring the effects of ST on other determinants of performance (i.e., VO2max, vVO2max, and sprint capacity). For example, it has been found that ST could induce a trivial effect on V O2max in endurance athletes [15]. In addition, the incorporation of diverse ST methods has demonstrated improvements in running economy among endurance runners [

# QA Chain

1. Question is applied to the vector store.
2. Vector store provides k relevant documents.
3. Docs and original question are sent to an LLM.

In [27]:
from langchain.chains import RetrievalQA  # not just langchain

# ⚠️ llm = ChatOpenAI(...)  → uses your OPENAI_API_KEY
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=compression_retriever,
    # optional, if you also want to see which docs were used
    return_source_documents=True  
)

# 💸 LLM call happens here
result = qa_chain.invoke({"query": question})  
print("Answer:", result["result"])
print("\n--- Source documents used ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print("Metadata:", doc.metadata)
    print("Content preview:", doc.page_content[:500])  # first 500 chars

Answer: I don't know.

--- Source documents used ---

Source 1:
Metadata: {'robots': 'noindex', 'source': 'pdfs/s40279-024-02018-z.pdf', 'crossmarkmajorversiondate': '2010-04-23', 'total_pages': 33, 'crossmarkdomainexclusive': 'true', 'author': 'Cristian Llanos-Lagos', 'page_label': '1802', 'creator': 'Springer', 'producer': 'PyPDF', 'page': 1, 'subject': 'Sports Medicine, https://doi.org/10.1007/s40279-024-02018-z', 'creationdate': '2024-04-16T17:19:55+05:30', 'moddate': '2024-07-10T21:13:03+05:30', 'crossmarkdomains[2]': 'springerlink.com', 'doi': '10.1007/s40279-024-02018-z', 'title': 'The Effect of Strength Training Methods on Middle-Distance and Long-Distance Runners’ Athletic Performance: A Systematic Review with Meta-analysis', 'crossmarkdomains[1]': 'springer.com'}
Content preview: Maximal oxygen consumption, velocity at maximal oxy-gen consumption, maximum metabolic steady state, and sprint capacity exhibited trivial changes after strength training. Indeed, VO2max has long bee

This did not work that well - now let's build our own prompt (a default was used):

In [32]:
from langchain.prompts import PromptTemplate

template = """Use the following pieces of context to answer the question.

#Context
{context}

#Question
{question}
"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# ⚠️ llm = ChatOpenAI(...)  → uses your OPENAI_API_KEY
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs = {"prompt": QA_CHAIN_PROMPT}
)

# 💸 LLM call happens here
result = qa_chain.invoke({"query": question})  
print("Answer:", result["result"])
print("\n--- Source documents used ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print("Metadata:", doc.metadata)
    print("Content preview:", doc.page_content[:500])  # first 500 chars

Answer: To improve your VO2max, consider the following strategies:

1. **Aerobic Training**: Engage in regular aerobic exercises such as running, cycling, swimming, or rowing. Aim for a mix of steady-state and interval training to enhance your cardiovascular fitness.

2. **High-Intensity Interval Training (HIIT)**: Incorporate HIIT sessions into your training regimen. These involve short bursts of intense exercise followed by rest or low-intensity periods, which can significantly boost VO2max.

3. **Increase Training Volume**: Gradually increase the duration and frequency of your aerobic workouts. This can help improve your overall endurance and cardiovascular capacity.

4. **Strength Training**: While strength training has shown trivial effects on VO2max, it can still contribute to overall performance improvements. Incorporate strength training to enhance your muscular endurance and efficiency.

5. **Optimize Running Economy**: Work on your running form and efficiency, as improvements